# 00 · Quickstart, step by step: from a simulation snapshot to the summary statistics

The second entry point to SkewerJAX: one TNG50-4 skewer (snapshot 25, z = 3), followed through every step of the
calculation, one figure per step, with the equations behind it:

1. **the simulation outputs** around the skewer;
2. **the cells it crosses** (the Voronoi mesh of the snapshot);
3. **the fields along it** (density, temperature, velocity);
4. **the line profiles** of the gas;
5. **τ and F** for Lyα and Si III λ1206, stretch by stretch, on one velocity axis, and the gas in a cylinder around
   the skewer;
6. **the overflux** and 7. **the summary statistics**.

Every function used here lives in the package (`skewerjax.simulations`, `skewerjax.core`, `skewerjax.utils.plotting`).
Runtime: ~2 min (CPU).

**Requires:** every ingredient of 01–07, on a TNG50-4 snapshot.  
**Independent of:** —

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
from skewerjax.core import voigt as V
from scipy.ndimage import gaussian_filter
from skewerjax.utils.plotting import NEON_COLORS as NC, NEON_CYCLE, dark_axes, neon_line, neon_legend, plot_slab, plot_tau_buildup
from skewerjax.core import LineBundle, wrap_to_ref_line, DeltaFieldBuilder, equivalent_width_kms
from skewerjax.core.stats import OverfluxStatistics1D, fit_mcdonald_a, mcdonald_factor
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, HI, Ion, slab_projection, tau_by_path_bins, tau_absorbers
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H, SI = reader.header, Ion("Si", 3)
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
LYA, LSI = LineBundle(["HI-LyA"], ref_line_id="HI-LyA"), LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
print(f"TNG50-4 snapshot 25: z = {H.redshift:.3f}, box {H.box_cMpc:.2f} cMpc, {H.n_gas:,} gas cells")

In [ ]:
plot_dag(('simulation', 'fields', 'tau', 'overflux', 'stats',)); plt.show()

## 1 · The simulation outputs

A TNG snapshot stores, for every gas cell of its Voronoi mesh, the position of its mesh-generating point, its mass
and density (so its volume V = M/ρ), its internal energy, electron abundance, metal mass fractions, neutral hydrogen
fraction, star-formation rate and velocity (notebook 05). `slab_projection` streams the snapshot and bins the cells
of a 1.5 cMpc thick slab into a surface-density map and a mass-weighted temperature map; the skewer runs along x
through the densest pixel of the smoothed map.

In [ ]:
z_c, thick = 0.5 * H.box_cMpc, 1.5
slab = slab_projection(reader, 3, z_c, thick, n_pix=400)
pix = H.box_cMpc / slab["sigma"].shape[0]
i_pk, j_pk = np.unravel_index(np.argmax(gaussian_filter(np.log10(np.maximum(slab["sigma"], 1e-2)), 2.0, mode="wrap")), slab["sigma"].shape)
x_pk, y_sk = (i_pk + 0.5) * pix, (j_pk + 0.5) * pix
sk = SimulationSkewers.axis_aligned(reader, np.array([[0.0, y_sk, z_c]]) / H.code_length_cMpc, [1], 0.25)
pv_H, pv_S = sk.point_values(HI), sk.point_values(SI)
s = sk.s[0]
print(f"slab: {slab['n_cells']:,} cells; skewer along x at y = {y_sk:.2f}, z = {z_c:.2f} cMpc through the peak at x = {x_pk:.2f}; "
      f"{len(sk.cells):,} cells selected")

In [ ]:
fig = plt.figure(figsize=(22, 12), layout="constrained")
gs = fig.add_gridspec(3, 2, width_ratios=[1.1, 1])
axm = fig.add_subplot(gs[:, 0]); axp = [fig.add_subplot(gs[i, 1]) for i in range(3)]
dark_axes([axm] + axp, fig)
im = plot_slab(axm, slab, smooth=1.0, label=r"log$_{10}\ \Sigma_{\rm gas}$ [M$_\odot$ ckpc$^{-2}$]")
neon_line(axm, [0, H.box_cMpc], [y_sk, y_sk], NC["HI"], lw=1.8)
ins = axm.inset_axes((0.62, 0.62, 0.36, 0.36))
plot_slab(ins, slab, smooth=1.0, colorbar=False, vmin=im.norm.vmin, vmax=im.norm.vmax); neon_line(ins, [0, H.box_cMpc], [y_sk, y_sk], NC["HI"], lw=1.4)
ins.set(xlim=(x_pk - 3, x_pk + 3), ylim=(y_sk - 3, y_sk + 3), xlabel="", ylabel=""); ins.tick_params(colors=FG, labelsize=16)
axm.indicate_inset_zoom(ins, edgecolor=FG)
axm.set_title(f"gas in |z − {z_c:.1f}| < {thick / 2} cMpc, and the skewer", color=FG)
neon_line(axp[0], s, pv_H["n"][0], NC["HI"], label="H I"); neon_line(axp[0], s, pv_S["n"][0], NC["SiIII"], label="Si III")
neon_line(axp[1], s, pv_H["T"][0], NC["temperature"]); neon_line(axp[2], s, pv_H["v_los"][0], NC["velocity"])
n_pos = pv_S["n"][0][pv_S["n"][0] > 0]
axp[0].set(yscale="log", ylabel=LABELS["n"], ylim=(np.percentile(n_pos, 10) / 100, None));   # clip the empty-cell dips of Si III
axp[1].set(yscale="log", ylabel=LABELS["T"]); axp[2].set(ylabel=LABELS["v_los"], xlabel=LABELS["s"])
for ax in axp:
    ax.axvline(x_pk, color=FG, ls=":", lw=1.2); ax.set_xlim(0, H.box_cMpc)
neon_legend(axp[0], loc="upper left", ncol=2)
dark_axes([axm] + axp, fig); fig.suptitle("(1, 3) The gas around a TNG50-4 skewer (left) and along it (right)", color=FG, fontsize=22)
plt.show()

The skewer crosses a filament at the dotted line: the ion densities rise by orders of magnitude, the gas is
shock-heated to ~10⁵–10⁶ K around it and falls in (the velocity changes sign across it).

## 2 · The cells the skewer crosses

Along a straight path every point belongs to the cell of its nearest mesh-generating point, so the gas is piecewise
constant: the path is a sequence of **cell crossings**, each of exact length ℓ_c (`SimulationSkewers.crossings()`).
The sampling points of the skewer are a separate grid: point i stands for its dual segment of length

$$w_i = \tfrac12\left(|x_{i+1} - x_i| + |x_i - x_{i-1}|\right),\qquad \sum_i w_i = \text{path length},$$

and the deposition cuts every dual segment into the pieces p of the cells it crosses (Σ_{p ∈ i} ℓ_p = w_i). The column
of the skewer, N = Σ_c n_c ℓ_c, is the same whatever the spacing of the points.

In [ ]:
cr = sk.crossings()
live = cr.cell[0] >= 0
lo, hi, cc = cr.s_lo[0][live], cr.s_hi[0][live], cr.cell[0][live]
size = np.asarray(sk.cells.size_cMpc)[cc]
nH_c = np.asarray(sk.properties(HI)["n_ion"])[cc]
fig, axs = plt.subplots(2, 1, figsize=(20, 8), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]}, layout="constrained")
norm = plt.matplotlib.colors.LogNorm(np.percentile(nH_c, 2), nH_c.max())
cmap = plt.get_cmap(CMAPS["diverging"])                                   # both ends bright on the dark background
axs[0].bar(0.5 * (lo + hi), hi - lo, width=hi - lo, color=cmap(norm(nH_c)), edgecolor=BG, lw=0.3)
axs[0].set(ylabel=r"crossed length $\ell_c$ [cMpc]", yscale="log", title=f"{live.sum()} cell crossings along the skewer, coloured by n_HI")
axs[1].semilogy(0.5 * (lo + hi), size, ".", color=TOL["cyan"], ms=4, label=r"cell size $V^{1/3}$")
axs[1].semilogy(0.5 * (lo + hi), hi - lo, ".", color=TOL["yellow"], ms=3, alpha=0.6, label=r"crossed length $\ell_c$")
axs[1].axhline(np.median(np.diff(sk.s[0])), color=FG, ls=":", lw=1.4, label="spacing of the sampling points (0.25 km/s)")
axs[1].set(xlabel=LABELS["s"], ylabel="[cMpc]"); axs[1].legend(fontsize=15, ncol=3)
fig.colorbar(plt.matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axs[0], label=r"$n_{\rm HI}$ [cm$^{-3}$]")
add_panel_letters(axs); fig.suptitle("The Voronoi cells crossed by the skewer: small in the filament, large in the voids", fontsize=21); plt.show()
print(f"column N_HI: from the crossings {np.sum(nH_c * (hi - lo)) * 3.085678e24 * H.a:.4e} cm^-2, from the deposited pieces "
      f"{sk.column_per_pixel(HI).sum():.4e} cm^-2")

## 3 · The fields along the skewer

What the integrator needs per piece of gas is the comoving density of the absorbing ion, its temperature and its
velocity along the photon. Per cell (notebook 05): n_H = ρX_H/(A_H m_u), T = (γ − 1)μ m_p u/k_B, x_HI from the stored
ionisation equilibrium (cold phase for star-forming cells), f_SiIII from the CLOUDY table, and n_ion = ρX_elem/(A m_u) ×
f_ion; v_∥ = v_pec · t̂. The profiles below are the segment averages of every dual segment (`point_values`).

## 4 · Line profiles

The gas at x absorbs the photon labelled by the observer's line-centred velocity v_† at the local wavelength

$$\lambda'_j(x) = \lambda_j\, e^{v_\dagger/c}\, \frac{a(x)}{a_\dagger}\, D^{-1}(x),\qquad \nu_j = c\Big(\frac{1}{\lambda'_j} - \frac{1}{\lambda_j}\Big),$$

with D the Doppler factor of its velocity along the photon. The profile is a Voigt function of that detuning,

$$\phi_{j}(\nu) = V(\nu; \sigma, \gamma) = \int G_\sigma(\nu')\, L_\gamma(\nu - \nu')\, d\nu',\qquad
\sigma = \frac{b}{\sqrt2\,\lambda_j},\ \ b = \sqrt{\frac{2k_BT}{m}},\qquad \gamma = \frac{A_{21}}{4\pi},$$

a Gaussian core of thermal width b and Lorentzian (natural) wings. Below: the profiles of the strongest piece of gas
on the skewer, in Lyα and in Si III (the same T; b_Si = 0.19 b_H because Si is 28 times heavier).

In [ ]:
from skewerjax.core import voigt_widths_GHz, phys
A_H, A_S = sk.absorbers(HI), sk.absorbers(SI)
p = int(np.argmax(A_H.n[0] * A_H.length[0]))
T_p = float(A_H.T[0, p])
dvv = np.linspace(-300, 300, 6001)
fig, ax = plt.subplots(figsize=(16, 7), layout="constrained")
for lid, ln in (("HI-LyA", lines.lines[0]), ("SiIII-1206", lines.lines[1])):
    sg, gm = voigt_widths_GHz(jnp.array([T_p]), jnp.zeros(1), jnp.zeros(1), jnp.array([ln.lambda0_A]), jnp.array([ln.A21_GHz]), jnp.array([ln.mass_amu]))
    nu = phys.c * 10.0 / ln.lambda0_A * np.expm1(-dvv / phys.c_kms)
    phi = np.asarray(V.voigt_exojax(jnp.asarray(nu), float(np.ravel(sg)[0]), float(np.ravel(gm)[0])))
    b = np.sqrt(2 * phys.kB * T_p / ln.mass_amu) / 1e3
    ax.semilogy(dvv, phi / phi.max(), color=NC[lid], label=f"{lid}: b = {b:.1f} km/s, γ λ = {float(np.ravel(gm)[0]) * ln.lambda0_A * 1e-4:.1e} km/s")
ax.set(xlabel=r"$v_\dagger - v_c$ [km/s]", ylabel=r"$\phi / \phi_{\max}$", ylim=(1e-7, 2),
       title=f"the Voigt profiles of the strongest piece (T = {T_p:.2e} K): Gaussian cores, Lorentzian wings")
ax.legend(fontsize=15); plt.show()

## 5 · Optical depth and flux, for Lyα and Si III

Summing the profiles of every piece p along the path, with its comoving density n_com,p, its length ℓ_p and its
scale factor a_p,

$$\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}}{a^{3}}\,\phi_j\, a\, dx \;\simeq\; \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\,\ell_p}{a_p^2}\, \phi_{j,p}(v_\dagger),
\qquad \mathcal{C}_j = \frac{\pi e^2}{m_e c} f_j,\qquad F_j = e^{-\tau_j}.$$

τ is linear in the densities, so the τ of each stretch of the path can be computed on its own and the stretches add
up to the full τ (`tau_by_path_bins`).

In [ ]:
fr = sk.frame("exact")
v, dv = sk.spectral_grid(1.0, fr)
A_H, A_S = sk.absorbers(HI), sk.absorbers(SI)
edges = np.linspace(0.0, H.box_cMpc, 9)
tb_H = tau_by_path_bins(A_H, LYA, v, fr, edges)[:, 0]
tb_S = tau_by_path_bins(A_S, LSI, v, fr, edges)[:, 0]
t_H, t_S = tau_absorbers(A_H, LYA, v, fr, check=None)[0, 0], tau_absorbers(A_S, LSI, v, fr, check=None)[0, 0]
print(f"sum over the 8 stretches - full tau: Lya {np.abs(tb_H.sum(0) - t_H).max() / t_H.max():.1e}, "
      f"Si III {np.abs(tb_S.sum(0) - t_S).max() / t_S.max():.1e} (relative, rounding)")

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(20, 15), layout="constrained", gridspec_kw={"height_ratios": [1.2, 1, 1, 1]})
dark_axes(axs, fig)
cols = plot_tau_buildup(axs[0], axs[1], edges, s, pv_H["n"][0], v, tb_H, line_label=r"Ly$\alpha$")
plot_tau_buildup(None, axs[2], edges, s, None, v, tb_S, colors=cols, line_label="Si III")
neon_line(axs[3], v, np.exp(-t_H), NC["HI-LyA"], label=r"Ly$\alpha$"); neon_line(axs[3], v, np.exp(-t_S), NC["SiIII-1206"], label="Si III 1206")
axs[1].set_yscale("symlog", linthresh=1e-2); axs[2].set_yscale("symlog", linthresh=1e-4)
axs[3].set(xlabel=LABELS["v_dagger"] + ", each line in its own frame", ylabel=LABELS["F"], ylim=(-0.05, 1.08))
for ax in axs[1:]:
    ax.set_xlim(v[0], v[-1])
axs[0].set_xlim(0, H.box_cMpc); axs[0].set_ylabel(r"$n_{\rm HI}\ [\mathrm{cm^{-3}}]$")
neon_legend(axs[1], loc="upper right"); neon_legend(axs[3], loc="lower left", ncol=2)
dark_axes(axs, fig); fig.suptitle("(5) Eight stretches of gas (top) and the absorption each one makes (below)", color=FG, fontsize=22)
plt.show()

The filament makes the saturated Lyα trough and nearly all of the Si III absorption, at the same line-centred
velocity in both lines. The diffuse stretches only add the shallow forest.

### The lines on one velocity axis

Each line is integrated in its own line-centred frame, so the same gas absorbs at the same v_† in every line (left).
An observer sees all lines on one wavelength axis. On the Lyα frame (`wrap_to_ref_line`), a feature of line j moves
by −Δv_j = −c ln(λ_Lyα/λ_j). The snapshot box is periodic in velocity (period P = N_v dv), so the shift wraps around.
The right panel tiles three periods and shows that the wrap is the same shift seen on the periodic tiling.

In [ ]:
tau_own = np.stack([t_H, t_S])[:, None]
tau_ref = np.asarray(wrap_to_ref_line(tau_own, v, lines))
P, dV = v.size * dv, float(lines.delta_v[1])
k_pk = int(np.argmax(t_S)); v_own = v[k_pk]; v_obs = np.mod(v_own - dV - v[0], P) + v[0]
fig, axs = plt.subplots(1, 2, figsize=(22, 7), layout="constrained", gridspec_kw={"width_ratios": [1, 1.6]})
dark_axes(axs, fig)
for j, (lid, t) in enumerate(zip(["HI-LyA", "SiIII-1206"], (t_H, t_S))):
    neon_line(axs[0], v, t / t.max(), NC[lid], label=lid)
axs[0].set(xlabel=LABELS["v_dagger"] + " (own frame)", ylabel=r"$\tau / \tau_{\max}$", title=r"same gas, same $v_\dagger$ (zoom)",
           xlim=(v_own - 300, v_own + 300))
tiles = np.arange(-1, 2)
for n in tiles:
    neon_line(axs[1], v + n * P, np.exp(-tau_ref[0, 0]), NC["HI-LyA"], label=r"Ly$\alpha$" if n == 0 else None, lw=1.2)
    neon_line(axs[1], v + n * P, np.exp(-tau_ref[1, 0]), NC["SiIII-1206"], label="Si III on the Ly$\\alpha$ frame" if n == 0 else None, lw=1.6)
    axs[1].axvspan(v[0] + n * P, v[-1] + n * P, color=FG, alpha=0.04 if n == 0 else 0.0)
    axs[1].axvline(v[0] + n * P, color=FG, ls=":", lw=1)
axs[1].annotate("", (v_own - dV, 0.12), (v_own, 0.12), arrowprops=dict(arrowstyle="-|>", color=FG, lw=2))
axs[1].text(v_own - dV / 2, 0.16, f"−Δv = −{dV:.0f} km/s", color=FG, ha="center", fontsize=15,
             bbox=dict(fc=BG, ec="none", alpha=0.85))
axs[1].plot([v_obs], [0.05], "v", color=NC["SiIII-1206"], ms=12)
axs[1].set(xlim=(v[0] - P, v[-1] + P), ylim=(-0.05, 1.12), xlabel=LABELS["v_ref"], ylabel=LABELS["F"],
           title=f"three periods (P = {P:.0f} km/s); the middle one is the box")
neon_legend(axs[0], loc="upper right"); neon_legend(axs[1], loc="lower right", ncol=2)
dark_axes(axs, fig); fig.suptitle("(5) Wrapping: Si III moves by −Δv on the Lyα frame, modulo the box period", color=FG, fontsize=22)
plt.show()
print(f"Si III peak: own frame {v_own:.0f} km/s -> Lya frame {v_obs:.0f} km/s = {v_own:.0f} - {dV:.0f} mod {P:.0f}")

### The gas in a cylinder, and the flux it absorbs

The skewer of (a)–(b) as the axis of a cylinder: the selected cells around it (the gas the deposition uses) glow by
their H I density, the H I and Si III densities along the skewer are drawn on the cylinder surface (log-scaled), and
the light crosses it from s = 0. `progress` is the fraction of the path already crossed: only the stretches before it
have absorbed (`tau_by_path_bins`, as in (b)), and the panel below shows the flux absorbed so far (bright) against the
final one (dim). One call per value of `progress` gives one frame of an animation; three stages are shown.

In [ ]:
from skewerjax.utils.plotting import plot_cylinder
P_cells = sk.properties(HI)
pos = np.asarray(sk.cells.positions)
d_perp = np.mod(pos[:, 1:] - np.array([y_sk, z_c]) + H.box_cMpc / 2, H.box_cMpc) - H.box_cMpc / 2      # (y, z) offsets from the axis
near = np.hypot(*d_perp.T) < 0.6
cloud = dict(axis=pos[near, 0], s_perp=d_perp[near, 0], t_perp=d_perp[near, 1], value=np.asarray(P_cells["n_ion"])[near])
edges_c = np.linspace(0.0, H.box_cMpc, 33)
tbc_H = tau_by_path_bins(A_H, LYA, v, fr, edges_c)[:, 0]
tbc_S = tau_by_path_bins(A_S, LSI, v, fr, edges_c)[:, 0]
s_of_v = np.interp(v, [v[0], v[-1]], [0.0, H.box_cMpc])                     # pixel -> path position (display: linear Hubble flow)
prof = {r"$\log n_{\rm HI}$": (np.log10(np.maximum(pv_H["n"][0], 1e-20)), NC["HI"]),
        r"$\log n_{\rm SiIII}$": (np.log10(np.maximum(pv_S["n"][0], 1e-24)), NC["SiIII"])}
flux_c = dict(v=v, s_of_v=s_of_v, edges=edges_c, tau_bins={r"Ly$\alpha$": (tbc_H, NC["HI-LyA"]), "Si III": (tbc_S, NC["SiIII-1206"])})
print(f"{near.sum():,} of {len(pos):,} selected cells within 0.6 cMpc of the axis; 32 stretches of {edges_c[1]:.2f} cMpc")
print(f"the filament of (a) is at s = {x_pk:.1f} cMpc, progress {x_pk / H.box_cMpc:.2f}")
for p in (0.1, 0.25, 1.0):
    fig, (ax3, axf) = plot_cylinder(s, prof, cloud=cloud, flux=flux_c, progress=p, radius=0.6, figsize=(22, 12))
    fig.suptitle(f"(e) progress = {p:.1f}: the light has crossed {p * H.box_cMpc:.1f} of {H.box_cMpc:.1f} cMpc", color=FG, fontsize=22, y=0.995)
    plt.show()

At progress 0.1 the light has not yet reached the filament of (a) (printed above): only the diffuse gas of
the first stretches has absorbed, a shallow forest. At 0.25 it has crossed the filament: the saturated Lyα trough, its
damping wings and the Si III absorption appear at once, at the same line-centred velocity. The remaining stretches add
the rest of the forest (1.0). The density profile on the cylinder and the absorption drawn at s(v) line up because the
velocity axis is the Hubble flow of the path (drawn as linear here, for display)

## 6 · The overflux

The flux of line j is written around its mean, F_j = F̄_j (1 + δ_j), and the total flux of several lines is their
product on one velocity axis (the Lyα frame, `wrap_to_ref_line`):

$$F = \prod_{j} \bar F_j\,(1 + \delta_j),\qquad \delta = \frac{F}{\langle F\rangle} - 1 .$$

`DeltaFieldBuilder` gives δ_j of each line, δ of the product and its exact decomposition into subset terms
(notebook 07); `OverfluxStatistics1D` measures P1D and ξ1D of every term.

## 7 · The summary statistics

On 16 random skewers, with both lines on the Lyα frame:
- `DeltaFieldBuilder` gives the overflux of each line, of their product and the exact decomposition of the total;
- `OverfluxStatistics1D` gives P1D and ξ1D by line pair, and `fit_mcdonald_a` fits the Si III amplitude of the
  total P1D;
- `equivalent_width_kms` and `SimulationSkewers.column_per_pixel` give per-skewer summaries.

In this TNG50-4 snapshot the Si III of random skewers is very weak (⟨F⟩ = 1.0000 to four decimals): the fitted amplitude is ~0 and
the Lyα × Si III cross term is ~1e-7, below the noise of 16 skewers, so no peak shows at the line separation.
Notebook 07 shows the same tools on a toy model where the Si III peak is clear.

In [ ]:
rng = np.random.default_rng(7)
cofm16, axis16 = rng.uniform(0, 35000.0, (16, 3)), rng.integers(1, 4, 16)
sk16 = SimulationSkewers.axis_aligned(reader, cofm16, axis16, 0.25)
v16, tau16 = sk16.optical_depth(lines)
tau16_ref = wrap_to_ref_line(tau16, v16, lines)
dv16 = float(v16[1] - v16[0])
ov = DeltaFieldBuilder(tau16_ref, labels=["Lya", "SiIII"], key_joiner="+").build()
st = OverfluxStatistics1D(ov, dv16).compute_p1d().compute_xi1d()
k = np.asarray(st.k)[1:]; P_tot = np.asarray(st.Pk_total)[1:].real; P_ll = np.asarray(st.Pk_pair[("Lya", "Lya")])[1:].real
fit = fit_mcdonald_a(k, P_tot, P_ll, delta_v=dV, k_range=(0.0, 0.1))
EW = np.asarray(equivalent_width_kms(tau16, dv16)); NHI = sk16.column_per_pixel(HI).sum(-1)
print(f"<F>: Lya {np.exp(-tau16[0]).mean():.3f}, Si III {np.exp(-tau16[1]).mean():.5f} | median EW: Lya {np.median(EW[0]):.0f} km/s, "
      f"Si III {np.median(EW[1]):.2f} km/s | log10 N_HI per skewer: {np.round(np.log10(NHI), 1)} | McDonald a = {fit['a']:.4f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(24, 7), layout="constrained")
dark_axes(axs, fig)
i0 = int(np.argmax(EW[1]))
for lab, key, c in (("Lya", "Lya", NC["HI-LyA"]), ("Si III", "SiIII", NC["SiIII-1206"])):
    neon_line(axs[0], v16, ov.delta_terms[key][i0], c, label=rf"$\delta_{{\rm {lab}}}$", lw=1.2)
neon_line(axs[0], v16, ov.total[i0], NC["total"], label=r"$\delta_{\rm tot}$", lw=1.0)
axs[0].set(xlabel=LABELS["v_ref"], ylabel=LABELS["delta"], title=f"overflux of skewer {i0}")
neon_line(axs[1], k, k * P_ll / np.pi, NC["HI-LyA"], label=r"Ly$\alpha$ alone")
neon_line(axs[1], k, k * P_tot / np.pi, NC["total"], label="total", lw=1.2)
neon_line(axs[1], k, k * P_ll * mcdonald_factor(k, fit["a"], dV) / np.pi, NC["SiIII-1206"], label=f"McDonald, a = {fit['a']:.3f}", lw=1.0)
kp = k * P_tot / np.pi; inr = k <= 0.1
axs[1].set(xscale="log", yscale="log", xlim=(k[0], 0.1), ylim=(kp[inr].min() / 3, kp[inr].max() * 3), xlabel=LABELS["k"], ylabel=LABELS["kP"],
           title="P1D, 16 skewers")
lags = np.asarray(st.lags); xc = np.asarray(st.xi_pair[("Lya", "SiIII")]).real; sel = (lags >= 0) & (lags <= P / 2)
neon_line(axs[2], lags[sel], xc[sel], NC["SiIII-1206"], label=r"$\xi_{\rm Ly\alpha \times SiIII}$")
for d in sorted({abs(np.mod(dV + P / 2, P) - P / 2)}):
    axs[2].axvline(d, color=FG, ls=":", lw=1.5)
axs[2].set(xlabel=LABELS["lag"], ylabel=LABELS["xi"], title="cross term (dotted: |Δv| mod P)")
for ax in axs:
    neon_legend(ax, loc="best")
dark_axes(axs, fig); fig.suptitle("(6, 7) Overflux, P1D with the Si III model, and the Lyα × Si III correlation", color=FG, fontsize=22)
plt.show()